# Modeling — XGBoost (quantile 0.9)

Desain: `docs/superpowers/specs/2026-08-19-xgboost-modeling-design.md`.
Rencana: `docs/superpowers/plans/2026-08-19-xgboost-modeling.md`.

Notebook ini tipis dengan sengaja. Semua logika ada di `utils/walk_forward.py`,
`utils/model_common.py` dan `utils/model_xgboost.py`, supaya jalur skrip dan
jalur notebook tidak bisa berbeda.

**Desember 2025 terkunci** dan tidak dinilai di sini.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from utils import evaluation, model_xgboost as xgb
from utils import modeling_prep, walk_forward

df = pd.read_parquet(modeling_prep.MODEL_INPUT_FILE)
print(f"{len(df):,} rows x {df.shape[1]} columns")

## Benchmark

Satu putaran dua-fit di fold 5 dengan `DEFAULT_PARAMS`, untuk mengukur ongkos
sebelum 60 fit pencarian dijalankan dan melihat di ronde berapa early stopping
mendarat. Angkanya dicatat di `docs/hasil-modeling-xgb.md`.

In [ ]:
import resource
import time

split = walk_forward.prepare_fold(df, 5)
train, valid = split["train"], split["valid"]
fit_rows, es_rows = xgb.split_early_stopping(train)
print(f"train {len(train):,} rows -> fit {len(fit_rows):,} + tail {len(es_rows):,}")
print(f"valid {len(valid):,} rows")

fit_predict = xgb.make_fit_predict(dict(xgb.DEFAULT_PARAMS))
start = time.time()
prediction = fit_predict(train, valid)
elapsed = time.time() - start

peak_bytes = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss  # bytes on macOS
print(f"best_iteration {fit_predict.best_iterations[0]} of {xgb.MAX_ROUNDS}")
print(f"wall time {elapsed / 60:.1f} min (both fits)")
print(f"peak RSS  {peak_bytes / 1024 ** 3:.2f} GB")
print(f"prediction mean {prediction.mean():.2f}, max {prediction.max():.2f}")

## Pencarian hyperparameter

30 kandidat dari ruang 2.592 kombinasi, dinilai di fold 3 dan 5 dengan
pinball@0.9 gabungan. Hasil yang dilaporkan datang dari walk-forward lima fold
di bawah, bukan dari sini — menilai di fold yang memilih pemenang akan
optimistis.

In [ ]:
candidates = xgb.sample_search_space(xgb.N_CANDIDATES, seed=42)
search_results = xgb.run_search(df, candidates, folds=xgb.SEARCH_FOLDS,
                                checkpoint_path=xgb.SEARCH_FILE)
search_results.to_csv(xgb.SEARCH_FILE, index=False)
search_results.sort_values("pinball").head(10)

## Walk-forward final

Konfigurasi pemenang di kelima fold, melawan ketiga baseline naive pada baris
yang identik.

In [ ]:
best = xgb.select_best(search_results, candidates)
xgb.save_best_params(best)
print(best)

fit_predict = xgb.make_fit_predict(best)
results = walk_forward.run_walk_forward(df, fit_predict, model_name="xgboost")
results.to_csv(xgb.RESULTS_FILE, index=False)

print("best_iteration per fold:", fit_predict.best_iterations)

overall = results[results["group_col"].isna()]
overall.pivot_table(index="model", columns="fold_id", values="pinball").round(3)

In [ ]:
bundle = xgb.fit_final(df, best)
xgb.save_bundle(bundle)
print(f"trained on {bundle['n_train']:,} rows, "
      f"{len(bundle['columns'])} columns, encoding {bundle['encoding']}, "
      f"{bundle['best_iteration']} rounds, quantile {bundle['quantile']}")

## Hasil

Tiga potongan, masing-masing melawan ketiga baseline naive pada baris identik.
Satu angka global menyesatkan di data yang 44% targetnya nol.

In [ ]:
results = pd.read_csv(xgb.RESULTS_FILE)

print("=== per fold (overall) ===")
print(results[results["group_col"].isna()]
      .pivot_table(index="model", columns="fold_id", values="pinball").round(3))

for group_col in walk_forward.GROUP_COLS:
    print(f"\n=== per {group_col} (pooled over folds) ===")
    grouped = results[results["group_col"] == group_col]
    table = (grouped.assign(weighted=grouped["pinball"] * grouped["n"])
                    .groupby(["model", "group_value"], observed=True)
                    .apply(lambda part: part["weighted"].sum() / part["n"].sum())
                    .unstack())
    print(table.round(3))

print("\n=== coverage and fill rate (overall, pooled) ===")
for model in results["model"].unique():
    print(f"{model:20s} "
          f"coverage {walk_forward.pooled_metric(results, model, 'coverage'):6.3f}  "
          f"fill_rate {walk_forward.pooled_metric(results, model, 'fill_rate'):6.3f}  "
          f"shortfall {walk_forward.pooled_metric(results, model, 'shortfall_units'):9.1f}")

## Head-to-head lawan Random Forest

Sah dilakukan karena kedua model dinilai di baris yang identik — dijamin
`walk_forward.eligible_rows()`, bukan oleh disiplin. Fold 1, 2, 4 adalah
potongan yang bersih: keduanya memilih pemenang di fold 3 dan 5.

In [ ]:
from utils import model_random_forest as rf

rf_results = pd.read_csv(rf.RESULTS_FILE)
combined = pd.concat([results, rf_results], ignore_index=True)

for label, folds in (("semua fold", None), ("fold 1/2/4 (bersih)", (1, 2, 4))):
    print(f"=== {label} ===")
    for model in ("xgboost", "random_forest", "naive_roll_mean_7"):
        rows = combined[(combined["model"] == model) & combined["group_col"].isna()]
        if rows.empty:
            continue
        print(f"{model:20s} "
              f"pinball {walk_forward.pooled_metric(combined, model, 'pinball', folds):6.3f}  "
              f"mae {walk_forward.pooled_metric(combined, model, 'mae', folds):7.3f}  "
              f"coverage {walk_forward.pooled_metric(combined, model, 'coverage', folds):6.3f}")
    print()